<a href="https://colab.research.google.com/github/vineetsalar88/ResearchPaper2/blob/master/TrainTestSplitterFromcsv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import pandas as pd
import cv2

# Paths
csv_path = "/content/drive/MyDrive/ResearchData/March26/annotations634to1128WithPath.csv"
image_dir = "/content/drive/MyDrive/ResearchData/March26/634to1128Dataset"
output_dir = "/content/drive/MyDrive/ResearchData/March26/634to1128croppedimages"

os.makedirs(output_dir, exist_ok=True)

# Load CSV
df = pd.read_csv(csv_path)

# Loop through each row
for idx, row in df.iterrows():
    img_name = row['image_name']
    xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
    label = str(row['label'])

    img_path = os.path.join(image_dir, img_name)

    # Read image
    image = cv2.imread(img_path)

    if image is None:
        print(f"Skipping missing image: {img_name}")
        continue

    h, w, _ = image.shape

    # Clamp bounding boxes (avoid errors)
    xmin = max(0, xmin)
    ymin = max(0, ymin)
    xmax = min(w, xmax)
    ymax = min(h, ymax)

    # Crop
    crop = image[ymin:ymax, xmin:xmax]

    # Create label folder
    label_dir = os.path.join(output_dir, label)
    os.makedirs(label_dir, exist_ok=True)

    # Save cropped image
    crop_filename = f"{os.path.splitext(img_name)[0]}_{idx}.jpg"
    save_path = os.path.join(label_dir, crop_filename)

    cv2.imwrite(save_path, crop)

print("✅ Cropping completed!")

✅ Cropping completed!


In [3]:

import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = "/content/drive/MyDrive/ResearchData/Dec25/AnnotatedDataset28Dec25/625Images1Jan26.csv"
image_dir = "/content/drive/MyDrive/ResearchData/Jan26/CroppedImages1Jan26"

train_dir = "/content/drive/MyDrive/ResearchData/Jan26/DatasetTrainVsTest1Jan/train"
val_dir = "/content/drive/MyDrive/ResearchData/Jan26/DatasetTrainVsTest1Jan/val"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

df = pd.read_csv(csv_path)
train_indices, val_indices = train_test_split(
    df.index,
    test_size=0.3,
    random_state=42,
    stratify=df["label"]   # keeps class balance
)
# ---- TRAIN ----
for idx in train_indices:
    row = df.iloc[idx]

    img_name = row["image_name"]
    label = str(row["label"])

    src = os.path.join(image_dir, img_name)

    dst_class = os.path.join(train_dir, label)
    os.makedirs(dst_class, exist_ok=True)

    dst = os.path.join(dst_class, img_name)

    shutil.copy(src, dst)

# ---- VALIDATION ----
for idx in val_indices:
    row = df.iloc[idx]

    img_name = row["image_name"]
    label = str(row["label"])

    src = os.path.join(image_dir, img_name)

    dst_class = os.path.join(val_dir, label)
    os.makedirs(dst_class, exist_ok=True)

    dst = os.path.join(dst_class, img_name)

    shutil.copy(src, dst)

KeyboardInterrupt: 

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import xml.etree.ElementTree as ET
import csv

# Input / Output files
xml_file = "/content/drive/MyDrive/ResearchData/AnnonatedDataset2March26/annotations.xml"
csv_file = "/content/drive/MyDrive/ResearchData/AnnonatedDataset2March26/annotations.csv"

# Parse XML
tree = ET.parse(xml_file)
root = tree.getroot()

# Collect all data
rows = []
max_boxes = 0

for image in root.findall('image'):
    image_data = {
        "id": image.get("id"),
        "name": image.get("name"),
        "width": image.get("width"),
        "height": image.get("height")
    }

    boxes = []
    for i, box in enumerate(image.findall('box')):
        box_data = {
            f"box/{i}/_label": box.get("label"),
            f"box/{i}/_source": box.get("source", ""),
            f"box/{i}/_occluded": box.get("occluded", ""),
            f"box/{i}/_xtl": box.get("xtl"),
            f"box/{i}/_ytl": box.get("ytl"),
            f"box/{i}/_xbr": box.get("xbr"),
            f"box/{i}/_ybr": box.get("ybr"),
            f"box/{i}/_z_order": box.get("z_order", ""),
            f"box/{i}/_rotation": box.get("rotation", "")
        }
        boxes.append(box_data)

    max_boxes = max(max_boxes, len(boxes))
    image_data["boxes"] = boxes
    rows.append(image_data)

# Build header dynamically
base_fields = ["id", "name", "width", "height"]
box_fields = [
    "_label", "_source", "_occluded", "_xtl", "_ytl", "_xbr", "_ybr", "_z_order", "_rotation"
]
header = base_fields + [f"box/{i}/{field}" for i in range(max_boxes) for field in box_fields]

# Write CSV
with open(csv_file, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()

    for row in rows:
        row_dict = {k: row[k] for k in base_fields}
        for i, box in enumerate(row["boxes"]):
            for field in box_fields:
                key = f"box/{i}/{field}"
                row_dict[key] = box.get(f"box/{i}/{field}", "")
        writer.writerow(row_dict)

print(f"✅ Flattened CSV saved as '{csv_file}' with up to {max_boxes} boxes per image.")

✅ Flattened CSV saved as '/content/drive/MyDrive/ResearchData/AnnonatedDataset2March26/annotations.csv' with up to 1 boxes per image.
